# Note: Run code on a GPU
## T4 via Colab works well


In [ ]:
#Install required packages
!pip install pymongo h3 requests pandas
!pip install srai
!pip install srai[osm]
!pip install h3
!pip install pytorch_lightning
!pip install hdbscan
!pip install torch
!pip install contextily
!pip install minisom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 35.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.1/173.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 114.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB

In [ ]:
# --- 1. System & Warning Suppression ---
import os
import warnings
import logging

# Filter specific annoying warnings that clutter client outputs
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', FutureWarning)
warnings.simplefilter('ignore', UserWarning)
warnings.simplefilter('ignore', DeprecationWarning)
os.environ["PYTHONWARNINGS"] = "ignore" # Suppress at system level

# Silence Pytorch Lightning & Library logs
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("srai").setLevel(logging.WARNING)

# --- 2. Data Manipulation & Math ---
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
import itertools

# --- 3. Geospatial & H3 ---
import h3
from srai.loaders import OSMPbfLoader
from srai.regionalizers import geocode_to_region_gdf, H3Regionalizer
from srai.joiners import IntersectionJoiner
from srai.embedders import Hex2VecEmbedder
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders.osm_loaders.filters import HEX2VEC_FILTER # This filter is used for loading specific OSM data
from srai.plotting import plot_regions

# --- 4. Machine Learning & Torch ---
import torch
from pytorch_lightning.callbacks import TQDMProgressBar
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import hdbscan
from sklearn.metrics import adjusted_rand_score, silhouette_score

# --- 5. Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import matplotlib.colors as mcolors

# --- 6. Configuration ---
# Set Matmul precision for Torch
torch.set_float32_matmul_precision('medium')

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('mode.chained_assignment', None) # Suppress SettingWithCopyWarning

# Matplotlib inline
%matplotlib inline

print("Libraries loaded. Environment configured for clean output.")

Libraries loaded. Environment configured for clean output.


#Set up MongoDB

In [ ]:
# 1. Fix the APT repo sync error & Install System Dependencies
# We use -o Acquire::Retries=3 and ignore the failing CRAN repo to let the rest install
!apt-get update -o Acquire::Languages=none
!apt-get install -y --fix-missing zstd gnupg wget pciutils lshw

# 2. Install MongoDB 7.0 (Official Repository)
!wget -qO- https://www.mongodb.org/static/pgp/server-7.0.asc | gpg --dearmor | tee /usr/share/keyrings/mongodb-server-7.0.gpg > /dev/null
!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-7.0.list
!apt-get update
!apt-get install -y mongodb-org

# 3. Start MongoDB Service
!mkdir -p /data/db
import subprocess
# If mongod is already running from a previous failed attempt, this will just handle it
subprocess.Popen(["mongod", "--fork", "--logpath", "/var/log/mongodb.log", "--dbpath", "/data/db"])

# 4. Install Ollama (using the manual method to ensure it lands in the right spot)
!curl -fsSL https://ollama.com/install.sh | sh

# 5. Start Ollama in the background with a slight delay
import time
import os

ollama_path = "/usr/local/bin/ollama"

if os.path.exists(ollama_path):
    with open("ollama_server.log", "w") as log_file:
        subprocess.Popen([ollama_path, "serve"], stdout=log_file, stderr=log_file)
    print("🚀 Ollama server starting...")
    time.sleep(12)
    !ollama pull qwen2.5
else:
    print("❌ Ollama installation failed. check the output of the curl command above.")

# 6. Final verification
print("\n--- Diagnostic Check ---")
!mongosh --eval "db.adminCommand('ping')"
!ollama list

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,995 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,913 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,293 kB]
Hit:13 https://ppa.launchpadcontent.net/graphi

In [ ]:
# Check MongoDB
!mongosh --eval "db.adminCommand('ping')"

# Check Ollama
!ollama list

]0;mongosh mongodb://127.0.0.1:27017/?directConnection=true&serverSelectionTimeoutMS=2000{ ok: 1 }
]0;NAME              ID              SIZE      MODIFIED     
qwen2.5:latest    845dbda0ea48    4.7 GB    1 second ago    


In [ ]:
# =====================================================
# 2.1 GLOBAL REPRODUCIBILITY SETUP
# =====================================================
# Setting seeds ensures that the Neural Network (Hex2Vec)
# and other stochastic processes produce the exact same
# results every time you run this notebook.

import random
import pytorch_lightning as pl

def set_global_seed(seed=42):
    # 1. Python's built-in random module
    random.seed(seed)

    # 2. NumPy (used for mathematical operations and arrays)
    np.random.seed(seed)

    # 3. PyTorch (used by Hex2Vec for weights initialization)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # 4. PyTorch Lightning (manages the training loop)
    # This is the most important one for the SRAI library
    pl.seed_everything(seed, workers=True)

# Apply the seed
set_global_seed(42)
print("Global random seed set to 42 for reproducibility.")

INFO:lightning_fabric.utilities.seed:Seed set to 42


Global random seed set to 42 for reproducibility.


# Set area of interest and H3 resolution

In [ ]:
area_of_interest = "Blacksburg, VA"
H3_resolution = 10

# Generate feature vectors

In [ ]:
# 1. Define Region
area = geocode_to_region_gdf(area_of_interest)

# 2. Load OSM Features
loaderpbf = OSMPbfLoader()
loadergdf = loaderpbf.load(area, HEX2VEC_FILTER)

# 3. H3 Regionalization
regionalizer = H3Regionalizer(resolution = H3_resolution)
regions_gdf = regionalizer.transform(loadergdf)

# 4. Join Features to H3 Regions
joiner = IntersectionJoiner()
joint_gdf = joiner.transform(regions_gdf, loadergdf)
#joint_gdf.head()
loadergdf.head()
#regions_gdf.head()

#print the type of a random entry in loadergdf
print(type(loadergdf.iloc[5][5]))

# Get the list of feature columns from HEX2VEC_FILTER keys
feature_columns = list(HEX2VEC_FILTER.keys())

# Apply the conversion to 0 or 1 for each feature column
for col in feature_columns:
    # Use .loc for column-wise assignment to avoid SettingWithCopyWarning
    # and ensure the whole column is processed efficiently.
    loadergdf.loc[:, col] = loadergdf[col].notna().astype(int)

loadergdf.head()
#len(loadergdf) : 15408

  0%|                                              | 0.00/53.7M [00:00<?, ?B/s]

Output()

Finished operation in 0:00:55

<class 'NoneType'>


,geometry,aeroway,amenity,building,healthcare,historic,landuse,leisure,military,natural,office,shop,sport,tourism,water,waterway
feature_id,,,,,,,,,,,,,,,,
relation/1589880,"MULTIPOLYGON (((-80.5824 37.18598, -80.58208 3...",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
way/429016822,"POLYGON ((-80.42826 37.19321, -80.42685 37.193...",0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
way/426064889,"POLYGON ((-80.42242 37.18964, -80.42327 37.189...",0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
way/1335106630,"POLYGON ((-80.41856 37.19883, -80.41859 37.198...",0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
way/1335106632,"POLYGON ((-80.41862 37.19895, -80.41866 37.198...",0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Sum up the features for each hex region

# Get the list of feature columns from HEX2VEC_FILTER keys
feature_columns = list(HEX2VEC_FILTER.keys())
joined_features_df = joint_gdf.reset_index()
merged_df = joined_features_df.merge(
    loadergdf[feature_columns], # Select only the feature columns from loadergdf
    left_on='feature_id',
    right_index=True, # Merge on loadergdf's index ('feature_id')
    how='left'
)
#len(merged_df) : 38785
# 3. Group by region_id and sum the feature columns
features_per_hex = merged_df.groupby('region_id')[feature_columns].sum()

print(f"Shape of features_per_hex: {features_per_hex.shape}")
print("First 5 rows of features_per_hex:")
print(features_per_hex.head())

# Print the row of features_per_hex corresponding to hexagon
# resolution 8: 882a8ac655fffff
# resoluteion 9: 892a8ac655bffff
# resolution 10: 8a2a8ac655a7fff
print("\nFeatures for hexagon 8a2a8ac655a7fff:")
print(features_per_hex.loc['8a2a8ac655a7fff'])


Shape of features_per_hex: (14152, 15)
First 5 rows of features_per_hex:
                aeroway amenity building healthcare historic landuse leisure  \
region_id                                                                      
8a2a8a890007fff       0       0        0          0        0       0       0   
8a2a8a89000ffff       0       0        0          0        0       0       0   
8a2a8a890017fff       0       0        0          0        0       0       0   
8a2a8a89001ffff       0       0        0          0        0       0       0   
8a2a8a890027fff       0       0        0          0        0       0       0   

                military natural office shop sport tourism water waterway  
region_id                                                                  
8a2a8a890007fff        0       1      0    0     0       0     0        0  
8a2a8a89000ffff        0       1      0    0     0       0     0        0  
8a2a8a890017fff        0       1      0    0     0       0    

# Set up RAG functions

In [ ]:
from pymongo import MongoClient, UpdateOne
import pandas as pd

def setup_nosql_and_ingest(features_per_hex: pd.DataFrame, db_name="hex_rag", collection_name="hex_features"):
    # 1. Connect to MongoDB
    client = MongoClient("mongodb://localhost:27017/")
    db = client[db_name]
    collection = db[collection_name]

    # 2. Prepare data
    # Note: Ensure 'region_id' is actually your index name or a column
    df_to_insert = features_per_hex.reset_index()
    records = df_to_insert.to_dict(orient="records")

    if records:
        # 3. Use Bulk Write for efficiency
        # We assume the H3 index column is named 'region_id' in your dataframe
        operations = [
            UpdateOne({"region_id": r["region_id"]}, {"$set": r}, upsert=True)
            for r in records
        ]
        collection.bulk_write(operations)
        print(f"✅ MongoDB Synced: {len(records)} hexes.")

    # 4. Ensure index for fast retrieval
    collection.create_index("region_id", unique=True)
    return collection

def retrieve_neighborhood_context(collection, target_hex: str, rings: int = 1):
    # Get hexes within n rings of target hex
    neighborhood_hexes = list(h3.grid_disk(target_hex, rings))

    # Query MongoDB for all hexes in the set; exclude the internal _id from results
    results = list(collection.find({"region_id": {"$in": neighborhood_hexes}}, {"_id": 0}))

    # Structure the final output to distinguish the center from the surroundings
    context = {"target": None, "neighbors": []}

    # Categorize hexes as the target or as neighbors
    for row in results:
        if row["region_id"] == target_hex:
            context["target"] = row
        else:
            context["neighbors"].append(row)
    return context

# Creates the prompt to be sent to LLM
def build_prompt(target_hex: str, context: dict):
    # Catch invalid target hexes
    if not context["target"]: return "No data found."

    # Build first part of prompt
    prompt = f"Analyze H3 Hexagon {target_hex} in Blacksburg, VA.\n\n### Local Features:\n"
    for k, v in context["target"].items():
        if k != "region_id" and v > 0: prompt += f"- {k}: {v}\n"

    if context["neighbors"]:
        # Add features of neighbor hexes
        prompt += "\n### Nearby Context (Aggregated):\n"
        agg = {}
        for n in context["neighbors"]:
            for k, v in n.items():
                if k != "region_id" and isinstance(v, (int, float)):
                    agg[k] = agg.get(k, 0) + v
        for k, v in agg.items():
            if v > 0: prompt += f"- Nearby {k}: {v}\n"
    # Add end of prompt where we ask for features of hex and surrounding area
    prompt += "\nTask: Based on these OSM features, describe the character of this neighborhood."
    #print("The prompt is :" + prompt + "\n")
    return prompt

# Send prompt to LLM
def generate_insights(prompt: str):
    response = requests.post("http://localhost:11434/api/generate",
                             json={"model": "qwen2.5", "prompt": prompt, "stream": False})
    return response.json().get("response", "Error generating response.")

# Calls the component functions of RAG pipeline
def run_rag_pipeline(df, query_hex, rings=1):
    col = setup_nosql_and_ingest(df)
    ctx = retrieve_neighborhood_context(col, query_hex, rings)
    prompt = build_prompt(query_hex, ctx)
    print("\n--- INSIGHTS ---\n", generate_insights(prompt))

# Load Ollama

In [ ]:
import subprocess
import time
import requests

# 1. Kill any ghost processes if they exist
!pkill ollama

# 2. Start Ollama server and pipe output to a log file
ollama_path = "/usr/local/bin/ollama"
with open("ollama_server.log", "w") as log_file:
    subprocess.Popen([ollama_path, "serve"], stdout=log_file, stderr=log_file)

# 3. Wait and "Ping" the server until it answers
print("Waiting for Ollama to wake up...")
for i in range(20):  # Try for 20 seconds
    try:
        # Port 11434 is the default Ollama API port
        response = requests.get("http://localhost:11434/")
        if response.status_code == 200:
            print("✅ Ollama is officially UP and running!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
        if i == 19:
            print("❌ Ollama failed to start. Run '!cat ollama_server.log' to see why.")

Waiting for Ollama to wake up...
✅ Ollama is officially UP and running!


# Call pipeline

In [ ]:
# Select a sample hex from your dataframe
test_hex = features_per_hex.index[0]

# Run the full pipeline
run_rag_pipeline(features_per_hex, query_hex=test_hex, rings=1)

✅ MongoDB Synced: 14152 hexes.

--- INSIGHTS ---
 Based on the OpenStreetMap (OSM) feature "H3 Hexagon 8a2a8a890007fff" in Blacksburg, Virginia, and the provided local and nearby context, we can infer certain characteristics about the neighborhood.

### Analysis:

1. **H3 Hexagon Code:**
   - The H3 hexagon code `8a2a8a890007fff` is a geographic identifier used to uniquely pinpoint an area on Earth's surface with high resolution. This specific code corresponds to a small, well-defined region within Blacksburg.

2. **Local Feature (natural: 1):**
   - The local feature indicates that there is at least one natural element present in the immediate vicinity of the H3 hexagon. This could suggest proximity to green spaces like parks, forests, or bodies of water, which are common in residential areas and can enhance the quality of life for residents.

3. **Nearby Context (Aggregated - Nearby natural: 6):**
   - The aggregated feature "nearby natural" with a value of 6 suggests that there are 

## Verify hex features

In [ ]:
features_per_hex.index[0]
features_per_hex.head(1)

,aeroway,amenity,building,healthcare,historic,landuse,leisure,military,natural,office,shop,sport,tourism,water,waterway
region_id,,,,,,,,,,,,,,,
8a2a8a890007fff,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


## Plot hex to visually validate outputs

In [ ]:
import h3
import folium
from shapely.geometry import Polygon

def plot_h3_hex(h3_address):
    # 1. Get the latitude/longitude of the hexagon center
    lat, lng = h3.cell_to_latlng(h3_address)

    # 2. Get the coordinates for the hexagon boundary
    # h3.cell_to_boundary returns (lat, lng) tuples
    boundary = h3.cell_to_boundary(h3_address)

    # 3. Create a Folium map centered on the hexagon
    # Using 'CartoDB dark_matter' for high contrast, as seen in your local files
    m = folium.Map(location=[lat, lng], zoom_start=15, tiles='CartoDB dark_matter')

    # 4. Add the hexagon to the map
    # Note: Folium expects [lat, lng], which matches cell_to_boundary output
    folium.Polygon(
        locations=boundary,
        color='cyan',
        weight=3,
        fill=True,
        fill_color='cyan',
        fill_opacity=0.4,
        tooltip=f"H3 Address: {h3_address}"
    ).add_to(m)

    return m

# Example Usage:
h3_id = features_per_hex.index[0]
map_view = plot_h3_hex(h3_id)
map_view

In [ ]:
test_hex = features_per_hex.index[1321]

run_rag_pipeline(features_per_hex, query_hex=test_hex, rings=1)

✅ MongoDB Synced: 14152 hexes.

--- INSIGHTS ---
 To analyze the OSM (OpenStreetMap) features provided for H3 Hexagon `8a2a8a896cc7fff` in Blacksburg, VA, we need to consider the given landuse data and nearby context. Here's a breakdown:

### Local Features:
- **Landuse: 1** - This could indicate that the hexagon is primarily used for residential purposes or has mixed-use areas with predominantly residential buildings.

### Nearby Context (Aggregated):
- **Nearby Landuse: 6** - The aggregated landuse value of 6 suggests a mix of different land uses in the surrounding area, possibly including commercial, industrial, and/or recreational spaces.
- **Nearby Natural: 1** - This indicates that there is some natural landscape or green space nearby, which could include parks, forests, or other open areas.

### Character Analysis:
Based on these features, we can infer that the neighborhood around H3 Hexagon `8a2a8a896cc7fff` in Blacksburg, VA has a residential core with mixed-use characteristic

In [ ]:
h3_id = features_per_hex.index[1321]
map_view = plot_h3_hex(h3_id)
map_view